# プロンプト圧縮 統合評価システム

## 機能概要
- **5種類の圧縮手法**をプラグイン形式で切り替え可能
  - `TFIDFCompressor` : TF-IDFによる文抽出（軽量・高速）
  - `SelectiveContext` : 自己情報量ベースのトークン削減
  - `LLMLingua` : Microsoftの小型LLMによる圧縮
  - `LongLLMLingua` : 長文向けLLMLinguaの拡張版
  - `AttentionPruning` : Attentionスコアベースの重要語抽出
- **評価メトリクス** : 圧縮率・ROUGE・BERTScore・レイテンシ・総合スコア
- **セマンティックキャッシュ** : 時間減衰＋コストスコアリング付き
- **可視化ログ** : 全判断過程をリアルタイム表示

## セル1: 環境セットアップ

In [ ]:
# === 依存パッケージのインストール ===
!apt-get -qq install redis-server
!pip install -q redis sentence-transformers transformers scikit-learn rouge-score

# bert_scoreはオプション（重い場合はスキップ可）
try:
    import bert_score
    HAS_BERT_SCORE = True
except ImportError:
    !pip install -q bert-score
    HAS_BERT_SCORE = True

print("✅ パッケージインストール完了")

## セル2: 基盤クラスのインポートと初期化

In [ ]:
import subprocess, time, json, math, re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import redis
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

# Redisサーバー起動
subprocess.Popen(["redis-server"])
time.sleep(1)
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 共有モデル（遅延ロード）
_embedder = None
_llm_tokenizer = None
_llm_model = None

def get_embedder():
    global _embedder
    if _embedder is None:
        print("[モデルロード] SentenceTransformer...")
        _embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    return _embedder

def get_llm():
    global _llm_tokenizer, _llm_model
    if _llm_model is None:
        print("[モデルロード] flan-t5-base...")
        _llm_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        _llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    return _llm_tokenizer, _llm_model

print("✅ 基盤クラス定義完了")

## セル3: 圧縮手法の定義（プラグイン形式）

In [ ]:
# ============================================================
# 圧縮結果データクラス
# ============================================================
@dataclass
class CompressionResult:
    original: str
    compressed: str
    method: str
    original_tokens: int
    compressed_tokens: int
    latency_ms: float
    ratio: float = field(init=False)

    def __post_init__(self):
        self.ratio = 1 - (self.compressed_tokens / max(self.original_tokens, 1))

    def __str__(self):
        return (
            f"[{self.method}]\n"
            f"  元トークン数 : {self.original_tokens}\n"
            f"  圧縮後トークン数: {self.compressed_tokens}\n"
            f"  圧縮率       : {self.ratio*100:.1f}%\n"
            f"  レイテンシ   : {self.latency_ms:.1f}ms\n"
            f"  圧縮後テキスト: {self.compressed[:80]}..."
        )


# ============================================================
# 基底クラス：すべての圧縮手法はこれを継承する
# ============================================================
class BaseCompressor(ABC):
    """プラグイン形式の圧縮器基底クラス"""

    @property
    @abstractmethod
    def name(self) -> str:
        pass

    @abstractmethod
    def _compress(self, text: str, ratio: float) -> str:
        """サブクラスで実装する圧縮ロジック"""
        pass

    def compress(self, text: str, ratio: float = 0.5) -> CompressionResult:
        """圧縮を実行し、メタデータ付きの結果を返す"""
        tok, _ = get_llm()
        original_tokens = len(tok.encode(text))

        start = time.time()
        compressed = self._compress(text, ratio)
        latency_ms = (time.time() - start) * 1000

        compressed_tokens = len(tok.encode(compressed))

        return CompressionResult(
            original=text,
            compressed=compressed,
            method=self.name,
            original_tokens=original_tokens,
            compressed_tokens=compressed_tokens,
            latency_ms=latency_ms
        )


# ============================================================
# 手法1: TF-IDF文抽出（最も軽量）
# ============================================================
class TFIDFCompressor(BaseCompressor):
    """
    TF-IDFスコアで重要文を選択。
    外部モデル不要で最速。短いプロンプトに向く。
    """
    @property
    def name(self): return "TF-IDF"

    def _compress(self, text: str, ratio: float) -> str:
        sentences = [s.strip() for s in re.split(r'[。.!?\n]', text) if s.strip()]
        if len(sentences) <= 1:
            # 文が1つしかない場合は単語レベルで削減
            words = text.split()
            keep = max(1, int(len(words) * (1 - ratio)))
            return ' '.join(words[:keep])

        keep_n = max(1, int(len(sentences) * (1 - ratio)))
        try:
            vectorizer = TfidfVectorizer()
            tfidf_matrix = vectorizer.fit_transform(sentences)
            scores = tfidf_matrix.sum(axis=1).A1
            top_indices = sorted(np.argsort(scores)[-keep_n:].tolist())
            return '。'.join([sentences[i] for i in top_indices])
        except Exception:
            return '。'.join(sentences[:keep_n])


# ============================================================
# 手法2: SelectiveContext（自己情報量ベース）
# ============================================================
class SelectiveContextCompressor(BaseCompressor):
    """
    各トークンの自己情報量（-log p）を計算し、
    情報量の低い（予測しやすい）トークンを削除する。
    LLMLinguaの前身手法に相当する実装。
    """
    @property
    def name(self): return "SelectiveContext"

    def _compress(self, text: str, ratio: float) -> str:
        import torch
        tok, mdl = get_llm()

        # トークン分割
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = mdl(**inputs, decoder_input_ids=inputs["input_ids"])
            logits = outputs.logits

        # 各トークンの自己情報量を計算
        probs = torch.softmax(logits[0], dim=-1)
        token_ids = inputs["input_ids"][0]
        self_info = []
        for i, tid in enumerate(token_ids):
            p = probs[i, tid].item()
            self_info.append(-math.log(p + 1e-10))

        # 情報量の高いトークンを保持
        n_keep = max(1, int(len(token_ids) * (1 - ratio)))
        threshold = sorted(self_info, reverse=True)[n_keep - 1]
        kept_ids = [tid for tid, si in zip(token_ids.tolist(), self_info) if si >= threshold]

        return tok.decode(kept_ids, skip_special_tokens=True)


# ============================================================
# 手法3: LLMLingua（小型LLMによるトークン確率ベース圧縮）
# ============================================================
class LLMLinguaCompressor(BaseCompressor):
    """
    小型LLMがトークンの条件付き確率を計算し、
    確率の高い（文脈から予測しやすい）トークンを削除する。
    SelectiveContextより文脈考慮が強い。

    ※本来はllmlingua packageを使うが、ここではflan-t5で近似実装
    　本番利用時は: from llmlingua import PromptCompressor を使用
    """
    @property
    def name(self): return "LLMLingua"

    def _compress(self, text: str, ratio: float) -> str:
        import torch
        tok, mdl = get_llm()

        sentences = [s.strip() for s in re.split(r'[。.!?\n]', text) if s.strip()]
        if len(sentences) <= 1:
            return TFIDFCompressor()._compress(text, ratio)

        # 各文の条件付きスコアを計算
        context = ""
        scored = []
        for sent in sentences:
            prompt = context + sent if context else sent
            inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=256)
            with torch.no_grad():
                outputs = mdl(**inputs, decoder_input_ids=inputs["input_ids"])
                logits = outputs.logits
            probs = torch.softmax(logits[0], dim=-1)
            token_ids = inputs["input_ids"][0]
            # 平均自己情報量をスコアとする
            avg_si = sum(-math.log(probs[i, tid].item() + 1e-10)
                        for i, tid in enumerate(token_ids)) / len(token_ids)
            scored.append((avg_si, sent))
            context = prompt

        keep_n = max(1, int(len(scored) * (1 - ratio)))
        # スコアの高い（情報量の多い）文を保持
        top = sorted(scored, key=lambda x: x[0], reverse=True)[:keep_n]
        # 元の順序に戻す
        top_sents = {s for _, s in top}
        return '。'.join([s for s in sentences if s in top_sents])


# ============================================================
# 手法4: LongLLMLingua（長文向け・クエリ重視圧縮）
# ============================================================
class LongLLMLinguaCompressor(BaseCompressor):
    """
    LLMLinguaの長文拡張。クエリとの関連度を優先的に考慮し、
    後半部分（結論・重要情報が多い）を重く評価する位置バイアスを追加。
    1000トークン超のプロンプトに向く。
    """
    @property
    def name(self): return "LongLLMLingua"

    def _compress(self, text: str, ratio: float, query: str = "") -> str:
        sentences = [s.strip() for s in re.split(r'[。.!?\n]', text) if s.strip()]
        if len(sentences) <= 1:
            return TFIDFCompressor()._compress(text, ratio)

        embedder = get_embedder()

        # クエリとの類似度（なければ全文の中心との類似度）
        if query:
            query_emb = embedder.encode(query)
        else:
            query_emb = embedder.encode(text[:200])

        sent_embs = embedder.encode(sentences)
        sims = sk_cosine([query_emb], sent_embs)[0]

        # 位置バイアス：後半の文を少し優遇（長文では結論が重要）
        n = len(sentences)
        position_bias = np.linspace(0.8, 1.2, n)
        final_scores = sims * position_bias

        keep_n = max(1, int(n * (1 - ratio)))
        top_indices = sorted(np.argsort(final_scores)[-keep_n:].tolist())
        return '。'.join([sentences[i] for i in top_indices])


# ============================================================
# 手法5: AttentionPruning（Attentionスコアベース）
# ============================================================
class AttentionPruningCompressor(BaseCompressor):
    """
    エンコーダのAttentionスコアを使い、
    他のトークンから多く参照されるトークン（重要語）を保持する。
    構造的な重要語抽出に強い。
    """
    @property
    def name(self): return "AttentionPruning"

    def _compress(self, text: str, ratio: float) -> str:
        import torch
        tok, mdl = get_llm()

        inputs = tok(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = mdl.encoder(**inputs, output_attentions=True)

        # 全レイヤー・全ヘッドのAttentionを平均
        attentions = torch.stack(outputs.attentions)  # [layers, batch, heads, seq, seq]
        avg_attention = attentions.mean(dim=[0, 1, 2])  # [seq, seq]
        # 各トークンが受け取る注意量の合計
        token_importance = avg_attention.sum(dim=0).numpy()

        token_ids = inputs["input_ids"][0].tolist()
        n_keep = max(1, int(len(token_ids) * (1 - ratio)))
        top_indices = sorted(np.argsort(token_importance)[-n_keep:].tolist())
        kept_ids = [token_ids[i] for i in top_indices]

        return tok.decode(kept_ids, skip_special_tokens=True)


# ============================================================
# コンプレッサーレジストリ（プラグイン管理）
# ============================================================
COMPRESSOR_REGISTRY = {
    "tfidf"      : TFIDFCompressor,
    "selective"  : SelectiveContextCompressor,
    "llmlingua"  : LLMLinguaCompressor,
    "longlingua" : LongLLMLinguaCompressor,
    "attention"  : AttentionPruningCompressor,
}

def get_compressor(name: str) -> BaseCompressor:
    if name not in COMPRESSOR_REGISTRY:
        raise ValueError(f"未知の圧縮手法: {name}\n利用可能: {list(COMPRESSOR_REGISTRY.keys())}")
    return COMPRESSOR_REGISTRY[name]()

print("✅ 圧縮手法定義完了")
print(f"   利用可能な手法: {list(COMPRESSOR_REGISTRY.keys())}")

## セル4: 評価メトリクスの定義

In [ ]:
# ============================================================
# 評価結果データクラス
# ============================================================
@dataclass
class EvaluationResult:
    method: str
    compression_ratio: float    # 0〜1（高いほど多く削減）
    rouge1: float               # 重要単語の保持率
    rougeL: float               # 文構造の保持率
    semantic_sim: float         # 意味の保持率（コサイン類似度）
    latency_ms: float           # 圧縮処理時間
    bert_score_f1: float = 0.0  # BERTScore（オプション）
    composite_score: float = 0.0  # 総合スコア

    def compute_composite(self):
        """
        総合スコア = 意味保持 × 0.4 + ROUGE-L × 0.3 + 圧縮率 × 0.2 + 速度スコア × 0.1
        速度スコア: 100ms以内なら1.0、1000ms以上なら0.0
        """
        speed_score = max(0.0, min(1.0, 1.0 - (self.latency_ms - 100) / 900))
        self.composite_score = (
            self.semantic_sim  * 0.40 +
            self.rougeL        * 0.30 +
            self.compression_ratio * 0.20 +
            speed_score        * 0.10
        )
        return self

    def summary(self):
        print(f"\n{'='*50}")
        print(f"  手法: {self.method}")
        print(f"  圧縮率       : {self.compression_ratio*100:.1f}%")
        print(f"  ROUGE-1      : {self.rouge1:.3f}")
        print(f"  ROUGE-L      : {self.rougeL:.3f}")
        print(f"  意味類似度   : {self.semantic_sim:.3f}")
        print(f"  レイテンシ   : {self.latency_ms:.1f}ms")
        if self.bert_score_f1:
            print(f"  BERTScore F1 : {self.bert_score_f1:.3f}")
        print(f"  ★総合スコア  : {self.composite_score:.3f}")
        print(f"{'='*50}")


# ============================================================
# 評価器クラス
# ============================================================
class CompressionEvaluator:
    def __init__(self, use_bert_score: bool = False):
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
        self.use_bert_score = use_bert_score

    def evaluate(self, result: CompressionResult) -> EvaluationResult:
        """圧縮結果を多角的に評価する"""
        original = result.original
        compressed = result.compressed

        # ROUGE スコア
        rouge_scores = self.rouge.score(original, compressed)
        r1 = rouge_scores['rouge1'].fmeasure
        rL = rouge_scores['rougeL'].fmeasure

        # 意味的類似度（コサイン）
        embedder = get_embedder()
        orig_emb = embedder.encode(original)
        comp_emb = embedder.encode(compressed)
        sim = float(np.dot(orig_emb, comp_emb) /
                   (np.linalg.norm(orig_emb) * np.linalg.norm(comp_emb) + 1e-10))

        eval_result = EvaluationResult(
            method=result.method,
            compression_ratio=result.ratio,
            rouge1=r1,
            rougeL=rL,
            semantic_sim=sim,
            latency_ms=result.latency_ms,
        )

        # BERTScore（オプション・重い）
        if self.use_bert_score:
            try:
                from bert_score import score as bs
                _, _, F1 = bs([compressed], [original], lang="ja", verbose=False)
                eval_result.bert_score_f1 = F1.mean().item()
            except Exception as e:
                print(f"[BERTScore スキップ] {e}")

        return eval_result.compute_composite()


print("✅ 評価器定義完了")

## セル5: セマンティックキャッシュ（時間減衰付き）

In [ ]:
# ============================================================
# 時間減衰タウの決定（クエリの性質に応じて）
# ============================================================
def get_tau(prompt: str) -> int:
    """プロンプトの種類に応じてキャッシュ有効時間を決定"""
    if any(kw in prompt for kw in ["ニュース", "速報", "今日", "最新", "現在"]):
        return 300      # 5分：時事情報は陳腐化が早い
    elif any(kw in prompt for kw in ["とは", "定義", "概念", "原理", "歴史"]):
        return 86400    # 24時間：概念的な質問は安定
    elif any(kw in prompt for kw in ["方法", "手順", "使い方", "how to"]):
        return 21600    # 6時間：手順系
    return 3600         # 1時間：デフォルト


def token_len(text: str) -> int:
    tok, _ = get_llm()
    return len(tok.encode(text))


def save_cache(prompt: str, response: str, emb: np.ndarray,
               compressor_name: str = "none"):
    """圧縮手法情報も含めてキャッシュに保存"""
    tau = get_tau(prompt)
    key = f"cache:{len(r.keys('cache:*'))}"

    r.set(key, json.dumps({
        "prompt"     : prompt,
        "response"   : response,
        "embedding"  : emb.tolist(),
        "timestamp"  : time.time(),
        "length"     : token_len(response),
        "compressor" : compressor_name,
        "tau"        : tau,
    }), ex=int(tau * 2))

    print(f"  [キャッシュ保存] key={key}, TTL={tau*2}s, 圧縮手法={compressor_name}")


def search_cache(prompt: str, threshold: float = 0.5) -> Optional[str]:
    """セマンティック類似度＋時間減衰＋価値スコアでキャッシュ検索"""
    now = time.time()
    tau = get_tau(prompt)
    embedder = get_embedder()
    query_emb = embedder.encode(prompt)

    print(f"  [キャッシュ検索] tau={tau}s, 閾値={threshold}")

    best_score = 0.0
    best_res = None

    for key in r.keys("cache:*"):
        raw = r.get(key)
        if not raw:
            continue
        data = json.loads(raw)
        emb = np.array(data["embedding"])

        # 三要素スコア計算
        sim   = float(np.dot(query_emb, emb) /
                     (np.linalg.norm(query_emb) * np.linalg.norm(emb) + 1e-10))
        age   = now - data["timestamp"]
        decay = np.exp(-age / tau)
        value = min(data["length"] / 100, 1.0)  # レスポンスの豊富さ

        score = sim * decay * value

        print(f"    {key}: sim={sim:.3f} decay={decay:.3f} value={value:.3f} "
              f"→ score={score:.3f} [手法:{data.get('compressor','?')}]")

        if score > best_score:
            best_score = score
            best_res = data["response"]

    if best_score >= threshold:
        print(f"  [キャッシュヒット ✅] best_score={best_score:.3f}")
        return best_res

    print(f"  [キャッシュミス ❌] best_score={best_score:.3f}")
    return None


print("✅ キャッシュシステム定義完了")

## セル6: メインパイプライン（圧縮 → LLM → キャッシュ）

In [ ]:
def call_llm(prompt: str) -> str:
    """LLMを呼び出す（実際のAPIに差し替え可能）"""
    print("  [LLM呼び出し 🤖]")
    tok, mdl = get_llm()
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = mdl.generate(**inputs, max_new_tokens=80)
    return tok.decode(outputs[0], skip_special_tokens=True)


def smart_llm(
    prompt: str,
    compressor_name: str = "tfidf",
    compression_ratio: float = 0.3,
    cache_threshold: float = 0.5,
    verbose: bool = True
) -> dict:
    """
    完全パイプライン:
    1. キャッシュ検索
    2. プロンプト圧縮（指定手法）
    3. LLM呼び出し
    4. キャッシュ保存

    Returns:
        dict: response, compressor_name, cache_hit, compression_result
    """
    print(f"\n{'█'*60}")
    print(f"  クエリ    : {prompt}")
    print(f"  圧縮手法  : {compressor_name}")
    print(f"  圧縮率目標: {compression_ratio*100:.0f}%削減")
    print(f"{'─'*60}")

    # Step 1: キャッシュ検索
    cached = search_cache(prompt, threshold=cache_threshold)
    if cached:
        return {
            "response": cached,
            "cache_hit": True,
            "compressor_name": "(cached)",
            "compression_result": None
        }

    # Step 2: プロンプト圧縮
    print(f"  [圧縮実行] 手法={compressor_name}")
    compressor = get_compressor(compressor_name)
    comp_result = compressor.compress(prompt, ratio=compression_ratio)
    print(comp_result)

    # Step 3: LLM呼び出し（圧縮後プロンプトで）
    response = call_llm(comp_result.compressed)

    # Step 4: キャッシュ保存（元のプロンプトのembeddingで保存）
    embedder = get_embedder()
    emb = embedder.encode(prompt)
    save_cache(prompt, response, emb, compressor_name=compressor_name)

    return {
        "response": response,
        "cache_hit": False,
        "compressor_name": compressor_name,
        "compression_result": comp_result
    }

print("✅ メインパイプライン定義完了")

## セル7: 評価ベンチマーク（複数手法を横断比較）

In [ ]:
def benchmark_compressors(
    texts: list[str],
    compressor_names: list[str] = None,
    ratio: float = 0.3,
    use_bert_score: bool = False
) -> list[EvaluationResult]:
    """
    複数テキスト × 複数手法のベンチマークを実行し、
    総合スコアで手法をランキングする。

    Args:
        texts: 評価用テキストのリスト
        compressor_names: 評価する圧縮手法名のリスト（None=全手法）
        ratio: 圧縮率
        use_bert_score: BERTScoreを使うか（重い）

    Returns:
        総合スコア降順のEvaluationResultリスト
    """
    if compressor_names is None:
        compressor_names = list(COMPRESSOR_REGISTRY.keys())

    evaluator = CompressionEvaluator(use_bert_score=use_bert_score)
    all_results: dict[str, list[EvaluationResult]] = {n: [] for n in compressor_names}

    print(f"\n{'🔬 ベンチマーク開始':=^60}")
    print(f"  テキスト数: {len(texts)}, 手法数: {len(compressor_names)}")
    print(f"  圧縮率: {ratio*100:.0f}%削減")

    for i, text in enumerate(texts, 1):
        print(f"\n  --- テキスト {i}/{len(texts)} ---")
        print(f"  {text[:60]}...")

        for name in compressor_names:
            try:
                compressor = get_compressor(name)
                comp_result = compressor.compress(text, ratio=ratio)
                eval_result = evaluator.evaluate(comp_result)
                all_results[name].append(eval_result)
                print(f"    [{name:15s}] score={eval_result.composite_score:.3f} "
                      f"sim={eval_result.semantic_sim:.3f} "
                      f"ratio={eval_result.compression_ratio*100:.1f}% "
                      f"{eval_result.latency_ms:.0f}ms")
            except Exception as e:
                print(f"    [{name}] エラー: {e}")

    # 各手法の平均スコアを計算してランキング
    print(f"\n{'📊 ベンチマーク結果':=^60}")
    summary_results = []
    for name, results in all_results.items():
        if not results:
            continue
        avg = EvaluationResult(
            method=name,
            compression_ratio=np.mean([r.compression_ratio for r in results]),
            rouge1=np.mean([r.rouge1 for r in results]),
            rougeL=np.mean([r.rougeL for r in results]),
            semantic_sim=np.mean([r.semantic_sim for r in results]),
            latency_ms=np.mean([r.latency_ms for r in results]),
            bert_score_f1=np.mean([r.bert_score_f1 for r in results]),
        ).compute_composite()
        summary_results.append(avg)

    summary_results.sort(key=lambda x: x.composite_score, reverse=True)

    print(f"\n  {'順位':<4} {'手法':<15} {'総合':>6} {'意味':>6} {'ROUGE-L':>7} {'圧縮率':>6} {'速度(ms)':>8}")
    print(f"  {'-'*55}")
    for rank, r in enumerate(summary_results, 1):
        print(f"  {rank:<4} {r.method:<15} {r.composite_score:>6.3f} "
              f"{r.semantic_sim:>6.3f} {r.rougeL:>7.3f} "
              f"{r.compression_ratio*100:>5.1f}% {r.latency_ms:>8.1f}")

    print(f"\n  🏆 推奨手法: {summary_results[0].method} (総合スコア: {summary_results[0].composite_score:.3f})")

    return summary_results

print("✅ ベンチマーク定義完了")

## セル8: 実行デモ

In [ ]:
# ============================================================
# デモ1: 基本的な圧縮 + キャッシュ動作確認
# ============================================================
print("\n" + "="*60)
print("デモ1: 基本動作確認")
print("="*60)

test_queries = [
    ("量子力学とは何ですか",         "tfidf",      0.3),
    ("量子物理学の基本原理とは",     "selective",  0.3),   # ← 上と類似→キャッシュヒット期待
    ("最新のAIニュースを教えて",     "llmlingua",  0.2),
    ("機械学習の学習方法を教えて",   "longlingua", 0.4),
    ("量子力学とは何ですか",         "attention",  0.3),   # ← 完全一致→キャッシュヒット確実
]

for query, method, ratio in test_queries:
    result = smart_llm(query, compressor_name=method, compression_ratio=ratio)
    status = "💾 キャッシュ" if result["cache_hit"] else "🤖 LLM"
    print(f"\n  {status} 回答: {result['response'][:100]}")

In [ ]:
# ============================================================
# デモ2: ベンチマーク比較
# ============================================================
print("\n" + "="*60)
print("デモ2: 圧縮手法ベンチマーク")
print("="*60)

benchmark_texts = [
    """量子力学は20世紀初頭に発展した物理学の理論体系であり、
    原子・電子・光子などの微小粒子の振る舞いを記述する。
    古典力学とは異なり、粒子の位置と運動量を同時に正確に決定することはできない。
    これをハイゼンベルクの不確定性原理という。
    また、粒子は波と粒子の二重性を持ち、観測前は確率波として存在する。""",

    """機械学習とは、コンピュータがデータから自動的に学習し、
    予測や判断を行う能力を獲得する技術です。
    教師あり学習では正解データを用いてモデルを訓練します。
    教師なし学習ではデータの構造を自律的に発見します。
    強化学習では試行錯誤を通じて最適な行動を学習します。""",
]

# 軽量な手法のみで比較（重い手法はコメントアウト）
rankings = benchmark_compressors(
    texts=benchmark_texts,
    compressor_names=["tfidf", "selective", "llmlingua", "longlingua"],
    ratio=0.3,
    use_bert_score=False  # Trueにするとより精密だが重い
)

In [ ]:
# ============================================================
# デモ3: 個別手法の直接比較
# ============================================================
print("\n" + "="*60)
print("デモ3: 圧縮結果の直接比較")
print("="*60)

sample_text = """大規模言語モデルはトランスフォーマーアーキテクチャに基づいており、
膨大なテキストデータで事前学習される。
プロンプトエンジニアリングはモデルの出力品質を向上させる技術であり、
指示の明確さや文脈の提供が重要な要素となる。
プロンプト圧縮はトークン数を削減しつつ意味情報を保持する手法であり、
APIコストの削減と推論速度の向上に貢献する。"""

print(f"\n元テキスト ({len(sample_text)}文字):")
print(sample_text)

evaluator = CompressionEvaluator(use_bert_score=False)

for name in ["tfidf", "selective", "llmlingua", "longlingua"]:
    try:
        comp = get_compressor(name)
        result = comp.compress(sample_text, ratio=0.4)
        eval_result = evaluator.evaluate(result)
        print(f"\n[{name}] 圧縮後:")
        print(f"  {result.compressed}")
        eval_result.summary()
    except Exception as e:
        print(f"[{name}] エラー: {e}")